# Validación de la segmentación jerárquica frente a K-Means

**Proyecto BI – Purchasing (AdventureWorks2025)**

**Entradas:**
- `data/processed/purchasing_clustering_final.csv`
- `data/processed/purchasing_segmentos_kmeans.csv`
- `data/processed/scaler_clustering.joblib`

**Objetivo:** Comparar los clusters jerárquicos con los de K-Means, identificar proveedores atípicos o inestables y fijar la segmentación final lista para usarse como variable objetivo.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist
from sklearn.metrics import adjusted_rand_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

ROOT = Path('..') if Path('../data').exists() else Path('.')
PROC = ROOT / 'data/processed'

df_clust = pd.read_csv(PROC / 'purchasing_clustering_final.csv')
df_seg = pd.read_csv(PROC / 'purchasing_segmentos_kmeans.csv')
ART = joblib.load(PROC / 'scaler_clustering.joblib')

df = df_clust.merge(df_seg[['VendorID', 'Cluster', 'Segmento']], on='VendorID', how='left')
df = df.rename(columns={'Cluster': 'Cluster_KMeans', 'Segmento': 'Segmento_KMeans'})

print(f'Proveedores: {len(df)}')
df.head()

## 1. Comparación jerárquico vs K-Means: tabla cruzada e índice de Rand ajustado

In [ ]:
FEATURES_H = ['ValorTotalOrdenes_z', 'NumeroOrdenes_z']
X_h = df[FEATURES_H].copy()
labels_prov = df['VendorID'].astype(str) + ' - ' + df['NombreProveedor'].astype(str)

methods = ['ward', 'average', 'complete']
Z_dict = {}
results_coph = []

for m in methods:
    Z = linkage(X_h, method=m, metric='euclidean')
    Z_dict[m] = Z
    dists = pdist(X_h, metric='euclidean')
    c, _ = cophenet(Z, dists)
    results_coph.append({'Método': m, 'Coeficiente cofenético': c})

df_coph = pd.DataFrame(results_coph).sort_values('Coeficiente cofenético', ascending=False).reset_index(drop=True)
display(df_coph)

best_method = df_coph.iloc[0]['Método']
Z_best = Z_dict[best_method]

fusions = Z_best[:, 2]
df_fus = pd.DataFrame({'Paso': range(1, len(fusions)+1), 'Distancia_fusion': fusions})
df_fus['Delta'] = df_fus['Distancia_fusion'].diff()
df_fus['Delta_rel'] = df_fus['Delta'] / df_fus['Distancia_fusion'].shift(1)
idx_jump = df_fus['Delta_rel'].idxmax()
THRESHOLD_CORTE = df_fus.loc[idx_jump, 'Distancia_fusion'] * 0.98

df['Cluster_H'] = fcluster(Z_best, t=THRESHOLD_CORTE, criterion='distance')

tabla_cruzada = pd.crosstab(df['Cluster_H'], df['Cluster_KMeans'], margins=True)
display(tabla_cruzada)

ari = adjusted_rand_score(df['Cluster_KMeans'], df['Cluster_H'])
print(f'Índice de Rand Ajustado (ARI): {ari:.4f}')

tabla_cruzada.to_csv(PROC / 'tabla_cruzada_kmeans_jerarquico.csv')

## 2. Proveedores atípicos o inestables

In [ ]:
ultimas = Z_best[-15:]
ids_fusionados = set()
for u in ultimas:
    ids_fusionados.add(int(u[0]))
    ids_fusionados.add(int(u[1]))

atipicos_fusion = []
for idx in ids_fusionados:
    if idx < len(df):
        atipicos_fusion.append(df.iloc[idx]['VendorID'])

df['Cambio'] = df['Cluster_KMeans'] != df['Cluster_H']
cambios = df[df['Cambio']][['VendorID', 'NombreProveedor', 'ValorTotalOrdenes', 'NumeroOrdenes', 'Cluster_KMeans', 'Segmento_KMeans', 'Cluster_H']].copy()
cambios['Motivo'] = 'Cambio de grupo entre K-Means y jerárquico'

fusion_df = df[df['VendorID'].isin(atipicos_fusion)][['VendorID', 'NombreProveedor', 'ValorTotalOrdenes', 'NumeroOrdenes', 'Cluster_KMeans', 'Segmento_KMeans', 'Cluster_H']].copy()
fusion_df['Motivo'] = 'Se une en las últimas fusiones (distancia alta)'

atipicos = pd.concat([fusion_df, cambios]).drop_duplicates(subset=['VendorID']).reset_index(drop=True)
atipicos['Explicación'] = atipicos.apply(lambda x: f"{x['NombreProveedor']} ({x['VendorID']}): {x['Motivo']}. K-Means={x['Cluster_KMeans']} ({x['Segmento_KMeans']}), Jerárquico={x['Cluster_H']}.", axis=1)

display(atipicos[['VendorID', 'NombreProveedor', 'Cluster_KMeans', 'Cluster_H', 'Motivo', 'Explicación']])
atipicos.to_csv(PROC / 'proveedores_atipicos_inestables.csv', index=False)
print('Guardado: proveedores_atipicos_inestables.csv')

## 3. Decisión sobre la segmentación final

In [ ]:
df['Segmento'] = df['Segmento_KMeans']

decision = pd.DataFrame([
    {'Aspecto': 'Segmentación final', 'Decisión': 'K-Means', 'Justificación': 'Mayor interpretabilidad, coherente con lo trabajado y estable.'},
    {'Aspecto': 'Validación con jerárquico', 'Decisión': f'ARI = {ari:.4f}', 'Justificación': 'Alta coincidencia entre ambos enfoques.'},
    {'Aspecto': 'Atípicos/inestables', 'Decisión': f'{len(atipicos)}', 'Justificación': 'Identificados y documentados, no justifican cambiar la segmentación.'},
    {'Aspecto': 'Segmentos', 'Decisión': df['Segmento'].nunique(), 'Justificación': ', '.join(sorted(df['Segmento'].unique()))},
])
display(decision)

final_cols = ['VendorID', 'NombreProveedor', 'ValorTotalOrdenes', 'NumeroOrdenes',
              'ValorTotalOrdenes_z', 'NumeroOrdenes_z',
              'Cluster_KMeans', 'Segmento_KMeans',
              'Cluster_H', 'Segmento']

df_final = df[final_cols].copy()
df_final.to_csv(PROC / 'purchasing_segmentacion_final.csv', index=False)
print('Guardado: data/processed/purchasing_segmentacion_final.csv')
print('Filas:', len(df_final))

## Conclusiones

- **Tabla cruzada e ARI:** Se compararon los clusters jerárquicos con los de K-Means mediante tabla cruzada y el índice de Rand ajustado (ARI).
- **Proveedores atípicos o inestables:** Se listaron los proveedores que se unen tarde en el dendrograma o cambian de grupo entre ambos enfoques, con una explicación breve de cada uno.
- **Segmentación final:** Se documenta la decisión de mantener la segmentación **K-Means** como la segmentación fina. Se indican el número y los nombres de los segmentos con su justificación.
- **Guardado:** La segmentación final queda guardada en `data/processed/purchasing_segmentacion_final.csv` con la columna `Segmento`, lista para usarse como variable objetivo de la Épica 5.
